# 🏦 Sequential Workflow with Custom Compliance Executor

## Overview

This notebook demonstrates mixing AI agents with **custom executors** in a financial services workflow. We'll create a loan advisory pipeline where:

1. **Financial Advisor Agent** - Provides personalized loan product recommendations
2. **Compliance Executor** - Custom code that adds required regulatory disclosures

### 💼 Industry Use Case: Loan Product Recommendations with Compliance

A customer asks about loan products. The workflow:
1. AI agent provides helpful product recommendations
2. Custom executor ensures all responses include required compliance disclaimers

### ⚠️ Important Financial Disclaimer
> **This notebook is for educational and demonstration purposes only.** Always consult licensed financial professionals for actual financial advice.

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Custom Executor** | Python class that processes conversation programmatically |
| **Handler Contract** | Accept `list[ChatMessage]`, emit updated list via `ctx.yield_output()` |
| **Mixed Pipeline** | Combine AI agents with deterministic code processors |

### Architecture

```
Customer Question
    ↓
[input-conversation]
    ↓
Financial Advisor Agent (recommendations)
    ↓
[to-conversation:advisor]
    ↓
Compliance Executor (adds disclaimers)
    ↓
[complete]
    ↓
Compliant Response with Disclaimers
```

## Prerequisites

- ✅ Azure AI Foundry configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Setup and Imports

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Any

from agent_framework import (
    Agent,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

PROJECT_ENDPOINT = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
print(f"✅ Environment loaded: {PROJECT_ENDPOINT is not None and MODEL_DEPLOYMENT is not None}")


## 2️⃣ Create Custom Compliance Executor

This custom executor adds required regulatory disclaimers to all financial advice responses. In real banking applications, this ensures:

- **Regulatory Compliance**: All customer communications include required disclosures
- **Consistency**: Same disclaimers added regardless of which agent responds
- **Audit Trail**: Programmatic guarantee that compliance was applied

### Handler Contract
```python
@handler
async def handler_name(self, conversation: list[ChatMessage], ctx: WorkflowContext) -> None:
    # Process conversation
    # Add compliance content
    await ctx.yield_output(updated_conversation)
```

In [ ]:
class ComplianceExecutor(Executor):
    """Custom executor that adds required financial disclaimers to all agent responses.
    
    In production banking applications, this ensures every customer interaction
    includes required regulatory disclosures and compliance statements.
    """
    
    COMPLIANCE_DISCLAIMER = """
---
⚠️ IMPORTANT DISCLOSURES:
• All loan products subject to credit approval and underwriting guidelines
• Annual Percentage Rate (APR) may vary based on creditworthiness and market conditions
• This is general information and not a binding offer or commitment to lend
• Past performance does not guarantee future results
• Please consult with a licensed financial advisor for personalized advice
• FDIC insured where applicable | Equal Housing Lender

*Your institution name here* | NMLS# 123456
---
"""

    @handler
    async def add_compliance(
        self, 
        agent_response: AgentExecutorResponse, 
        ctx: WorkflowContext[list[Message], list[Message]]
    ) -> None:
        """Process agent response and add compliance disclaimers.
        
        Args:
            agent_response: Response from the previous agent (AgentExecutorResponse)
            ctx: Workflow context for sending output
        """
        # Extract conversation from agent response
        conversation = agent_response.full_conversation
        
        if not conversation:
            await ctx.yield_output([Message(role="assistant", contents=["No conversation to process."])])
            return
        
        # Count messages for audit logging
        user_msgs = sum(1 for m in conversation if m.role == "user")
        assistant_msgs = sum(1 for m in conversation if m.role == "assistant")
        
        print(f"📋 Compliance Executor: Processing {user_msgs} user and {assistant_msgs} assistant messages")
        
        # Add compliance disclaimer as a final message
        compliance_message = Message(
            role="assistant",
            contents=[self.COMPLIANCE_DISCLAIMER]
        )
        
        final_conversation = list(conversation) + [compliance_message]
        print("✅ Compliance disclaimers added to response")
        
        await ctx.yield_output(final_conversation)

## 3️⃣ Build and Execute Loan Advisory Workflow

### Participant Order
1. **Financial Advisor Agent**: AI-powered loan product recommendations
2. **Compliance Executor**: Deterministic compliance disclaimer insertion

In [ ]:
async def run_loan_advisory_workflow() -> None:
    with AzureCliCredential() as credential:
        chat_client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        try:
            print("✅ Foundry Chat Client created")

            # Financial Advisor Agent - Provides loan product recommendations
            financial_advisor = Agent(
                client=chat_client,
                instructions=(
                    "You are a Financial Advisor at a retail bank. Help customers understand their loan options.\n"
                    "When discussing loans:\n"
                    "1. Explain available products (personal loans, auto loans, mortgages, HELOCs)\n"
                    "2. Discuss general eligibility factors (credit score, income, debt-to-income ratio)\n"
                    "3. Mention typical rate ranges (without making specific promises)\n"
                    "4. Be helpful but never make guarantees about approval or specific rates\n"
                    "Keep responses concise and educational."
                ),
                name="financial_advisor",
            )
            print("✅ Financial Advisor Agent created")

            # Build sequential workflow: financial_advisor -> compliance_executor
            compliance_executor = ComplianceExecutor(id="compliance")
            workflow = (
                WorkflowBuilder(start_executor=financial_advisor, output_from=[compliance_executor])
                .add_chain([financial_advisor, compliance_executor])
                .build()
            )
            print("✅ Workflow built: Financial Advisor → Compliance Executor")

            # Process customer question
            customer_question = "I'm looking at buying my first car. What kind of auto loan options do you have, and what do I need to qualify?"

            print(f"\n{'=' * 60}")
            print("💬 CUSTOMER INQUIRY")
            print("=" * 60)
            print(f"👤 Customer: {customer_question}")
            print("=" * 60)

            # Run and print final conversation (get_outputs collects ctx.yield_output values)
            events = await workflow.run(customer_question)
            outputs = events.get_outputs()

            if outputs:
                print("\n📄 RESPONSE WITH COMPLIANCE")
                print("=" * 60)
                messages: list[Message] | Any = outputs[0]
                for i, msg in enumerate(messages, start=1):
                    name = msg.author_name or ("assistant" if msg.role == "assistant" else "user")
                    role_emoji = "👤" if msg.role == "user" else "🏦"
                    print(f"\n{'-' * 60}")
                    print(f"{role_emoji} [{name.upper()}]")
                    print(f"{'-' * 60}")
                    print(f"{msg.text}")

            print("\n" + "=" * 60)
            print("✅ Loan advisory workflow complete!")
        finally:
            await chat_client.client.close()
            await chat_client.project_client.close()


## 4️⃣ Run the Loan Advisory Workflow

In [ ]:
await run_loan_advisory_workflow()

## 📝 Key Takeaways

### Why Custom Executors for FSI?

| Benefit | Description |
|---------|-------------|
| **Guaranteed Compliance** | Code-based disclaimers can't be "forgotten" by AI |
| **Audit Trail** | Deterministic processing for regulatory audits |
| **Separation of Concerns** | AI handles advice, code handles compliance |
| **Customizable** | Easily update disclaimers without retraining models |

### Industry Use Cases for Custom Executors

```python
# Compliance Executor - Add required disclosures
# PII Redactor - Remove sensitive data before logging
# Rate Limiter - Throttle customer interactions
# Audit Logger - Record all interactions for compliance
# Sentiment Checker - Flag negative interactions for review
```

### Production Considerations

- Custom executors should be **lightweight and focused**
- Add **error handling** for edge cases
- Consider **timeout handling** for complex processing
- **Test independently** before integration
- **Log processing** for debugging and compliance